# Eigenvalues & Eigenvectors

## Definition
For a square matrix A, an **eigenvector** `v` and its corresponding **eigenvalue** `λ` satisfy:

```
A @ v = λ * v
```

The eigenvector only changes in **scale** (not direction) when multiplied by A.

### Key Properties
- `np.linalg.eig(A)` — general case (eigenvalues may be complex)
- `np.linalg.eigh(A)` — symmetric matrices (eigenvalues always real, eigenvectors orthogonal)
- Eigenvalues can be real or complex numbers
- `n×n` matrix has exactly `n` eigenvalues (counted with multiplicity)
- Symmetric positive definite matrices have all positive eigenvalues

### AI/ML Applications
| Use Case | How eigenvalues/vectors are used |
|----------|----------------------------------|
| **PCA** | Eigenvectors of covariance matrix = principal components |
| **Spectral clustering** | Eigenvectors of graph Laplacian |
| **PageRank** | Dominant eigenvector of link matrix |
| **Stability analysis** | Eigenvalues of weight/Jacobian matrices |
| **Covariance analysis** | Eigenvalue decomposition of X^T X |

In [ ]:
import numpy as np

# Simple 2x2 eigendecomposition
A = np.array([[4, 2],
              [1, 3]], dtype=float)

eigenvalues, eigenvectors = np.linalg.eig(A)
print('Eigenvalues:', eigenvalues)
print('Eigenvectors (columns):\n', eigenvectors)

In [ ]:
# Verify: A @ v = lambda * v for each eigenpair
A = np.array([[4, 2],
              [1, 3]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A)

for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]   # i-th eigenvector is the i-th column!
    lam = eigenvalues[i]
    lhs = A @ v
    rhs = lam * v
    print(f'Eigenvalue {lam:.2f}: A@v = {np.round(lhs, 4)}, λv = {np.round(rhs, 4)}, match={np.allclose(lhs, rhs)}')

In [ ]:
# Symmetric matrix: use eigh (more numerically stable, returns real values)
# Covariance matrix is always symmetric positive semi-definite
rng = np.random.default_rng(42)
X = rng.standard_normal((100, 5))  # 100 samples, 5 features
C = X.T @ X / (100 - 1)            # sample covariance matrix

eigenvalues, eigenvectors = np.linalg.eigh(C)
print('Covariance eigenvalues:', np.round(eigenvalues, 4))
print('Eigenvectors shape:', eigenvectors.shape)  # (5, 5)

# Eigenvalues from eigh are sorted ascending; flip for PCA
eigenvalues = eigenvalues[::-1]
eigenvectors = eigenvectors[:, ::-1]
print('Sorted desc:', np.round(eigenvalues, 4))

In [ ]:
# PCA using eigendecomposition
rng = np.random.default_rng(42)
X = rng.standard_normal((200, 4))  # (n_samples, n_features)

# Center data
X_centered = X - X.mean(axis=0)

# Covariance matrix
C = X_centered.T @ X_centered / (len(X) - 1)

# Eigendecomposition
vals, vecs = np.linalg.eigh(C)
# Sort descending
idx = np.argsort(vals)[::-1]
vals, vecs = vals[idx], vecs[:, idx]

# Project to top-2 components
X_pca = X_centered @ vecs[:, :2]
print('PCA output shape:', X_pca.shape)  # (200, 2)
print('Explained variance ratio:', np.round(vals / vals.sum(), 4))

In [ ]:
# Eigenvalue decomposition: A = V @ diag(Lambda) @ V_inv
A = np.array([[3, 1], [0, 2]], dtype=float)
vals, vecs = np.linalg.eig(A)

Lambda = np.diag(vals)
V = vecs
V_inv = np.linalg.inv(V)

A_reconstructed = V @ Lambda @ V_inv
print('A reconstructed:\n', np.round(A_reconstructed, 8))
print('Matches original:', np.allclose(A_reconstructed, A))

In [ ]:
# Matrix power using eigendecomposition
# A^k = V @ diag(lambda^k) @ V_inv
A = np.array([[2, 1], [0, 3]], dtype=float)
vals, vecs = np.linalg.eig(A)
V = vecs
V_inv = np.linalg.inv(V)

k = 4
A_k_eigen = V @ np.diag(vals**k) @ V_inv
A_k_direct = np.linalg.matrix_power(A, k)

print(f'A^{k} via eigen:\n', np.round(A_k_eigen, 4))
print(f'A^{k} direct:\n', A_k_direct)
print('Match:', np.allclose(A_k_eigen, A_k_direct))

In [ ]:
# EXCEPTION: Non-square matrix has no eigenvalues
rect = np.ones((3, 2))
try:
    np.linalg.eig(rect)
except np.linalg.LinAlgError as e:
    print('LinAlgError:', e)

# eigh on non-symmetric matrix gives wrong results silently
A_nonsymm = np.array([[1.0, 2.0], [3.0, 4.0]])
# eigh only uses lower triangle, so results will be WRONG
vals_wrong = np.linalg.eigh(A_nonsymm)[0]
vals_correct = np.linalg.eig(A_nonsymm)[0]
print('eigh on non-symmetric (WRONG):', np.round(np.sort(vals_wrong.real), 4))
print('eig (CORRECT):', np.round(np.sort(vals_correct.real), 4))